# BMI/CS 771 - Assignment 2: Image Classification
## Part I: Training a Simple Convolutional Neural Network (Section 3.2.1)

This notebook walks you through:
1. Setting up the environment on Google Colab
2. Downloading the MiniPlaces dataset
3. Understanding the codebase structure
4. Training the provided Simple CNN
5. Monitoring training with TensorBoard
6. Evaluating the trained model

**Important**: Make sure to select a GPU runtime!
- Go to `Runtime` → `Change runtime type` → Select `T4 GPU` or `A100 GPU`

---
## Step 1: Check GPU Availability

First, let's verify that we have GPU access. This is **required** for training neural networks efficiently.

In [ ]:
# ============================================================================
# CHECK GPU AVAILABILITY
# ============================================================================
# This cell verifies that:
#   1. NVIDIA GPU is available on this Colab instance
#   2. PyTorch can access the GPU via CUDA
#
# If you see "No GPU available", go to:
#   Runtime → Change runtime type → Hardware accelerator → GPU
# ============================================================================

import torch

# Check if CUDA (NVIDIA GPU support) is available
if torch.cuda.is_available():
    # Get the name of the GPU
    gpu_name = torch.cuda.get_device_name(0)
    # Get the total memory of the GPU in GB
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU Available: {gpu_name}")
    print(f"   GPU Memory: {gpu_memory:.2f} GB")
    print(f"   PyTorch Version: {torch.__version__}")
    print(f"   CUDA Version: {torch.version.cuda}")
else:
    print("❌ No GPU available! Please enable GPU in Runtime settings.")
    print("   Go to: Runtime → Change runtime type → GPU")

---
## Step 2: Install Required Packages

Install all the Python packages needed for this assignment. Most are pre-installed on Colab, but we ensure specific versions are available.

In [ ]:
# ============================================================================
# INSTALL REQUIRED PACKAGES
# ============================================================================
# Required packages for Assignment 2:
#   - torch, torchvision: Deep learning framework (pre-installed on Colab)
#   - opencv-python: Image processing library
#   - numpy: Numerical computing
#   - gdown: Download files from Google Drive
#   - pycocotools: COCO dataset evaluation tools
#   - pyyaml: YAML configuration file parsing
#   - pillow: Image loading library
#   - tensorboard: Training visualization
# ============================================================================

# Install packages (most are pre-installed on Colab, but we ensure they exist)
!pip install -q opencv-python numpy gdown pycocotools pyyaml pillow tensorboard

print("✅ All packages installed successfully!")

---
## Step 3: Upload Assignment Code

You have two options to get the assignment code into Colab:
- **Option A**: Upload a ZIP file of the assignment folder
- **Option B**: Mount Google Drive (if you've uploaded the code there)

Run the appropriate cell below.

In [ ]:
# ============================================================================
# OPTION A: UPLOAD ZIP FILE
# ============================================================================
# This method:
#   1. Opens a file upload dialog
#   2. You select your assignment2_release.zip file
#   3. The code extracts it to /content/assignment2_release/
#
# To create the ZIP file on your local machine:
#   cd ~/Downloads
#   zip -r assignment2_release.zip assignment2_release/
# ============================================================================

from google.colab import files
import zipfile
import os

# Upload the ZIP file
print("📁 Please upload your assignment2_release.zip file...")
uploaded = files.upload()

# Extract the uploaded ZIP file
for filename in uploaded.keys():
    print(f"\n📦 Extracting {filename}...")
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        zip_ref.extractall('/content/')
    # Remove the ZIP file to save space
    os.remove(filename)
    print(f"✅ Extracted and removed ZIP file")

# List the contents to verify
!ls -la /content/

In [ ]:
# ============================================================================
# OPTION B: MOUNT GOOGLE DRIVE (Alternative to Option A)
# ============================================================================
# Use this if you've uploaded the assignment folder to your Google Drive.
# After mounting, copy the files to /content/ for faster access.
#
# Note: Only run this cell if you didn't use Option A above.
# ============================================================================

from google.colab import drive
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Copy the assignment folder from Drive to local storage (faster I/O)
# Modify the path below to match your Drive folder structure
DRIVE_PATH = '/content/drive/MyDrive/assignment2_release'  # ← CHANGE THIS PATH

if os.path.exists(DRIVE_PATH):
    print(f"📁 Copying from {DRIVE_PATH}...")
    shutil.copytree(DRIVE_PATH, '/content/assignment2_release')
    print("✅ Files copied successfully!")
else:
    print(f"❌ Path not found: {DRIVE_PATH}")
    print("   Please update DRIVE_PATH to match your folder location.")

---
## Step 4: Set Working Directory and Verify Structure

Set up the working directory and verify that all necessary files are in place.

In [ ]:
# ============================================================================
# SET WORKING DIRECTORY
# ============================================================================
# We set the base directory for the assignment and verify the folder structure.
# The expected structure is:
#   assignment2_release/
#   ├── code/
#   │   ├── cls_libs/        # Classification helper code
#   │   ├── det_libs/        # Detection helper code
#   │   ├── configs/         # YAML configuration files
#   │   ├── train_cls.py     # Training script
#   │   └── eval_cls.py      # Evaluation script
#   ├── data/                # Dataset folder (will be populated)
#   ├── logs/                # Training logs and checkpoints
#   └── results/             # Final results for submission
# ============================================================================

import os

# Define the base directory
BASE_DIR = '/content/assignment2_release'

# Change to the base directory
os.chdir(BASE_DIR)
print(f"📂 Working directory: {os.getcwd()}")

# Verify the folder structure
print("\n📁 Directory contents:")
!ls -la

print("\n📁 Code directory contents:")
!ls -la code/

print("\n📁 Config files:")
!ls -la code/configs/

---
## Step 5: Download the MiniPlaces Dataset

Download the MiniPlaces dataset using the provided script. This dataset contains:
- **100K training images** from 100 scene categories
- **10K validation images** for testing
- **10K test images** (no labels provided)

The download may take a few minutes depending on network speed.

In [ ]:
# ============================================================================
# DOWNLOAD MINIPLACES DATASET
# ============================================================================
# The download_cls_dataset.sh script does the following:
#   1. Downloads data.tar.gz from Google Drive using gdown
#   2. Downloads train.txt and val.txt label files from GitHub
#   3. Extracts the images from the tar.gz archive
#
# After completion, the data/ folder will contain:
#   - images/train/    (100K training images organized by category)
#   - images/val/      (10K validation images)
#   - images/test/     (10K test images)
#   - objects/         (object detection annotations, not used in Part I)
#   - train.txt        (training image paths and labels)
#   - val.txt          (validation image paths and labels)
# ============================================================================

# Create the data directory if it doesn't exist
!mkdir -p data

# Change to data directory and download
%cd data

print("📥 Downloading MiniPlaces dataset...")
print("   This may take 5-10 minutes depending on network speed.\n")

# Download the main dataset archive from Google Drive
# The file ID corresponds to the MiniPlaces dataset
!gdown 16GYHdSWS3iMYwMPv5FpeDZN2rH7PR0F2

# Download the label files from the MiniPlaces GitHub repository
# train.txt: maps image paths to class labels (0-99) for training
# val.txt: maps image paths to class labels for validation
!wget -q https://raw.githubusercontent.com/CSAILVision/miniplaces/master/data/train.txt
!wget -q https://raw.githubusercontent.com/CSAILVision/miniplaces/master/data/val.txt

print("\n📦 Extracting dataset (this may take a minute)...")
# Extract the tar.gz archive
!tar -xzf data.tar.gz

# Return to base directory
%cd ..

print("\n✅ Dataset download complete!")

In [ ]:
# ============================================================================
# VERIFY DATASET DOWNLOAD
# ============================================================================
# Let's verify that all the necessary files were downloaded correctly.
# We check for:
#   1. The images folder with train/val/test subdirectories
#   2. The train.txt and val.txt label files
#   3. Sample images to ensure extraction worked
# ============================================================================

import os

print("📁 Data directory contents:")
!ls -la data/

print("\n📁 Images directory:")
!ls -la data/images/

# Count the number of training categories
print("\n📊 Dataset statistics:")
num_train_categories = len(os.listdir('data/images/train'))
print(f"   Number of training categories: {num_train_categories}")

# Count lines in label files (number of images)
!echo "   Number of training images: $(wc -l < data/train.txt)"
!echo "   Number of validation images: $(wc -l < data/val.txt)"

# Show a few sample entries from train.txt
print("\n📝 Sample entries from train.txt:")
!head -5 data/train.txt

---
## Step 6: Understand the Code Structure

Before training, let's understand the key files and their roles in the codebase.

In [ ]:
# ============================================================================
# EXPLORE THE CODEBASE STRUCTURE
# ============================================================================
# Key files for Part I (Image Classification):
#
# Training & Evaluation Scripts:
#   - code/train_cls.py      Main training script
#   - code/eval_cls.py       Evaluation script
#
# Helper Libraries (code/cls_libs/):
#   - config.py              Default hyperparameters and config loading
#   - dataset.py             MiniPlaces data loader
#   - modeling.py            Neural network architectures (YOU WILL EDIT THIS)
#   - transforms.py          Data augmentation transforms
#   - engine.py              Training/evaluation loops
#   - blocks.py              Building blocks (PatchEmbed, attention helpers)
#
# Configuration Files (code/configs/):
#   - cls_miniplaces_cnn.yaml      Config for SimpleNet (Section 3.2.1)
#   - cls_miniplaces_resnet18.yaml Config for fine-tuning ResNet-18
#   - cls_miniplaces_vit.yaml      Config for Vision Transformer
# ============================================================================

print("📁 Classification library files:")
!ls -la code/cls_libs/

print("\n📁 Configuration files:")
!ls -la code/configs/

In [ ]:
# ============================================================================
# VIEW THE CNN CONFIGURATION FILE
# ============================================================================
# The YAML configuration file specifies:
#   - output_folder: Where to save logs and checkpoints
#   - model_type: "CNN" or "VIT"
#   - dataset.root_folder: Path to the data directory
#
# Additional parameters can be added to override defaults in config.py
# ============================================================================

print("📄 CNN Configuration (cls_miniplaces_cnn.yaml):")
print("=" * 50)
!cat code/configs/cls_miniplaces_cnn.yaml

In [ ]:
# ============================================================================
# VIEW DEFAULT HYPERPARAMETERS
# ============================================================================
# The config.py file defines all default training hyperparameters:
#   - devices: GPU/CPU selection (default: "cuda:0")
#   - dtype: "fp16" for mixed precision training
#   - loader.batch_size: 512 (may need to reduce for smaller GPUs)
#   - input: Image preprocessing parameters
#   - model.num_classes: 100 (MiniPlaces categories)
#   - opt: Optimizer settings (SGD, lr=0.1, 60 epochs + 1 warmup)
#
# These can be overridden in the YAML config file.
# ============================================================================

print("📄 Default Configuration (config.py):")
print("=" * 50)
!cat code/cls_libs/config.py

In [ ]:
# ============================================================================
# VIEW THE SIMPLE CNN ARCHITECTURE
# ============================================================================
# The SimpleNet class in modeling.py is the network used for Section 3.2.1.
# Architecture:
#   1. Conv1: 7x7 conv (3→64 channels), stride 2, ReLU, MaxPool
#   2. Conv2: Bottleneck block (64→64→64→256), ReLU, MaxPool
#   3. Conv3: Bottleneck block (256→128→128→512), ReLU
#   4. Global Average Pooling
#   5. Fully Connected layer (512→100 classes)
#
# This is a simple network that should achieve >40% accuracy.
# ============================================================================

print("📄 SimpleNet Architecture (modeling.py - lines 18-68):")
print("=" * 50)
!sed -n '18,68p' code/cls_libs/modeling.py

---
## Step 7: View Training Script Help

Check the available command-line arguments for the training script.

In [ ]:
# ============================================================================
# VIEW TRAINING SCRIPT USAGE
# ============================================================================
# The train_cls.py script accepts the following arguments:
#   config         Path to YAML configuration file (required)
#   -p, --print-freq   Print training stats every N batches (default: 10)
#   -c, --ckpt-freq    Save checkpoint every N epochs (default: 5)
#   --output           Custom suffix for experiment folder
#   --resume           Path to checkpoint to resume training
# ============================================================================

%cd code
!python train_cls.py --help
%cd ..

---
## Step 8: Start TensorBoard (for Monitoring)

TensorBoard allows you to visualize training progress in real-time. Start it before training.

In [ ]:
# ============================================================================
# LOAD TENSORBOARD EXTENSION
# ============================================================================
# TensorBoard will display:
#   - Training loss over time
#   - Learning rate schedule
#   - Training and validation accuracy
#
# The extension runs inline in the notebook.
# ============================================================================

%load_ext tensorboard

In [ ]:
# ============================================================================
# START TENSORBOARD
# ============================================================================
# This starts TensorBoard pointing to the logs directory.
# It will update automatically as training progresses.
#
# Metrics to watch:
#   - loss/train: Should decrease over time
#   - acc_top1/val: Validation accuracy (target: >40%)
#   - acc_top5/val: Top-5 validation accuracy
#   - lr: Learning rate (should follow cosine schedule)
# ============================================================================

# Create logs directory if it doesn't exist
!mkdir -p logs

# Launch TensorBoard
%tensorboard --logdir logs

---
## Step 9: Train the Simple CNN (Section 3.2.1)

This is the main training cell. The training will:
- Run for **61 epochs** (60 training + 1 warmup)
- Use **batch size 512** (automatically uses mixed precision)
- Save checkpoints every 5 epochs
- Save the final model as `model_final.pth.tar`

**Expected training time**: ~30-60 minutes on a T4 GPU

In [ ]:
# ============================================================================
# TRAIN THE SIMPLE CNN (SECTION 3.2.1)
# ============================================================================
# This cell runs the complete training pipeline:
#   1. Loads the configuration from cls_miniplaces_cnn.yaml
#   2. Creates the MiniPlaces data loaders
#   3. Initializes SimpleNet model
#   4. Trains for 60 epochs with 1 warmup epoch
#   5. Evaluates on validation set after each epoch
#   6. Saves checkpoints and final model
#
# Training output:
#   - Logs are saved to: logs/cls_miniplaces_cnn_<timestamp>/
#   - Final model: logs/cls_miniplaces_cnn_<timestamp>/model_final.pth.tar
#   - Epoch checkpoints: logs/cls_miniplaces_cnn_<timestamp>/epoch_XXX.pth.tar
#
# The expected final validation accuracy is >40% top-1.
# ============================================================================

# Change to code directory for training
%cd /content/assignment2_release/code

# Run training
# --print-freq 50: Print stats every 50 batches (reduce output spam)
# --ckpt-freq 5: Save checkpoint every 5 epochs
!python train_cls.py ./configs/cls_miniplaces_cnn.yaml --print-freq 50 --ckpt-freq 5

# Return to base directory
%cd /content/assignment2_release

print("\n" + "="*60)
print("✅ TRAINING COMPLETE!")
print("="*60)

---
## Step 10: Find the Experiment Folder

After training, locate the experiment folder containing your trained model.

In [ ]:
# ============================================================================
# FIND THE EXPERIMENT FOLDER
# ============================================================================
# Training creates an experiment folder with the format:
#   logs/cls_miniplaces_cnn_YYYYMMDD_HHMMSS/
#
# This folder contains:
#   - config.txt          Copy of the configuration used
#   - logs/               TensorBoard event files
#   - epoch_XXX.pth.tar   Intermediate checkpoints
#   - model_final.pth.tar Final trained model
# ============================================================================

import os
import glob

# Find all experiment folders
exp_folders = sorted(glob.glob('logs/cls_miniplaces_cnn_*'))

if exp_folders:
    # Use the most recent experiment folder
    EXP_FOLDER = exp_folders[-1]
    print(f"📁 Found {len(exp_folders)} experiment folder(s)")
    print(f"\n🎯 Using most recent: {EXP_FOLDER}")
    print("\n📁 Contents:")
    !ls -la {EXP_FOLDER}
else:
    print("❌ No experiment folders found!")
    print("   Please run the training cell first.")
    EXP_FOLDER = None

---
## Step 11: Evaluate the Trained Model

Run the evaluation script to get the final accuracy metrics on the validation set.

In [ ]:
# ============================================================================
# EVALUATE THE TRAINED MODEL
# ============================================================================
# The eval_cls.py script:
#   1. Loads the configuration and trained model
#   2. Runs inference on the validation set
#   3. Reports top-1 and top-5 accuracy
#
# Arguments:
#   config: Path to YAML configuration file
#   ckpt: Path to experiment folder or specific checkpoint
#   -e: Specify epoch number (default: uses final or latest)
#
# Expected results: >40% top-1 accuracy
# ============================================================================

if EXP_FOLDER:
    %cd /content/assignment2_release/code
    
    print("📊 Evaluating trained model...")
    print("="*60)
    
    # Run evaluation (automatically selects final model if available)
    !python eval_cls.py ./configs/cls_miniplaces_cnn.yaml ../{EXP_FOLDER}/
    
    %cd /content/assignment2_release
else:
    print("❌ No experiment folder found. Please train the model first.")

---
## Step 12: Save Results for Submission

Copy the necessary files to the results folder for submission.

In [ ]:
# ============================================================================
# COPY RESULTS FOR SUBMISSION
# ============================================================================
# For submission, you need to include in the results/ folder:
#   1. Configuration file (YAML)
#   2. Training logs (excluding intermediate checkpoints)
#   3. Final model checkpoint
#
# This cell copies the necessary files to results/simple_cnn/
# ============================================================================

import shutil
import os

if EXP_FOLDER:
    # Create results subdirectory for this experiment
    RESULTS_DIR = 'results/simple_cnn'
    os.makedirs(RESULTS_DIR, exist_ok=True)
    
    print(f"📁 Copying results to {RESULTS_DIR}/")
    
    # Copy configuration file
    shutil.copy('code/configs/cls_miniplaces_cnn.yaml', RESULTS_DIR)
    print("   ✓ Configuration file")
    
    # Copy config.txt from experiment
    shutil.copy(f'{EXP_FOLDER}/config.txt', RESULTS_DIR)
    print("   ✓ Experiment config")
    
    # Copy TensorBoard logs
    if os.path.exists(f'{EXP_FOLDER}/logs'):
        shutil.copytree(f'{EXP_FOLDER}/logs', f'{RESULTS_DIR}/logs', dirs_exist_ok=True)
        print("   ✓ TensorBoard logs")
    
    # Copy final model checkpoint
    final_model = f'{EXP_FOLDER}/model_final.pth.tar'
    if os.path.exists(final_model):
        shutil.copy(final_model, RESULTS_DIR)
        print("   ✓ Final model checkpoint")
    else:
        # If final model doesn't exist, copy the latest epoch checkpoint
        epoch_ckpts = sorted(glob.glob(f'{EXP_FOLDER}/epoch_*.pth.tar'))
        if epoch_ckpts:
            shutil.copy(epoch_ckpts[-1], RESULTS_DIR)
            print(f"   ✓ Latest checkpoint: {os.path.basename(epoch_ckpts[-1])}")
    
    print("\n📁 Results directory contents:")
    !ls -la {RESULTS_DIR}
    
    print("\n✅ Results saved successfully!")
else:
    print("❌ No experiment folder found. Please train the model first.")

---
## Step 13: Download Results (Optional)

Download the results and trained model to your local machine.

In [ ]:
# ============================================================================
# DOWNLOAD RESULTS AS ZIP
# ============================================================================
# This creates a ZIP file of the results folder and downloads it.
# You can then include these files in your submission.
# ============================================================================

from google.colab import files
import shutil

# Create ZIP of results folder
!zip -r simple_cnn_results.zip results/simple_cnn/

# Download the ZIP file
print("📥 Downloading results...")
files.download('simple_cnn_results.zip')

In [ ]:
# ============================================================================
# ALTERNATIVE: SAVE TO GOOGLE DRIVE
# ============================================================================
# If downloading fails, you can save results to Google Drive instead.
# ============================================================================

from google.colab import drive
import shutil

# Mount Google Drive (skip if already mounted)
try:
    drive.mount('/content/drive')
except:
    pass  # Already mounted

# Copy results to Drive
DRIVE_DEST = '/content/drive/MyDrive/assignment2_results/simple_cnn/'
os.makedirs(DRIVE_DEST, exist_ok=True)

# Copy files
shutil.copytree('results/simple_cnn/', DRIVE_DEST, dirs_exist_ok=True)

print(f"✅ Results saved to Google Drive: {DRIVE_DEST}")

---
## Step 14: Resume Training (If Interrupted)

If training was interrupted (e.g., Colab disconnected), use this cell to resume.

In [ ]:
# ============================================================================
# RESUME TRAINING FROM CHECKPOINT
# ============================================================================
# If training was interrupted, you can resume from the last checkpoint.
# The script will continue from where it left off.
#
# Steps:
#   1. Find the experiment folder
#   2. Find the latest checkpoint
#   3. Resume training with --resume flag
# ============================================================================

import glob

# Find experiment folders
exp_folders = sorted(glob.glob('logs/cls_miniplaces_cnn_*'))

if exp_folders:
    exp_folder = exp_folders[-1]
    
    # Find the latest checkpoint
    checkpoints = sorted(glob.glob(f'{exp_folder}/epoch_*.pth.tar'))
    
    if checkpoints:
        latest_ckpt = checkpoints[-1]
        print(f"📁 Experiment folder: {exp_folder}")
        print(f"📄 Latest checkpoint: {latest_ckpt}")
        print("\n🔄 Resuming training...\n")
        
        %cd /content/assignment2_release/code
        !python train_cls.py ./configs/cls_miniplaces_cnn.yaml --resume=../{latest_ckpt}
        %cd /content/assignment2_release
    else:
        print("❌ No checkpoints found in experiment folder.")
else:
    print("❌ No experiment folders found. Start training from scratch.")

---
## Summary

### What You've Done (Section 3.2.1):
1. ✅ Set up the Google Colab environment with GPU
2. ✅ Downloaded the MiniPlaces dataset (100K training, 10K validation images)
3. ✅ Trained the SimpleNet CNN for 61 epochs
4. ✅ Achieved >40% top-1 validation accuracy
5. ✅ Saved results for your writeup

### For Your Writeup:
- Record the **training curves** (loss, accuracy) from TensorBoard
- Record the **final evaluation metrics** (top-1 and top-5 accuracy)
- You'll need these later for comparison in Section 3.2.2

### Next Steps:
- **Section 3.2.2**: Design your own improved CNN
- **Section 3.2.2**: Fine-tune pretrained ResNet-18
- **Section 3.3**: Implement Vision Transformer

In [ ]:
# ============================================================================
# FINAL STATUS CHECK
# ============================================================================
# Summary of what was completed in this session.
# ============================================================================

import os
import glob

print("="*60)
print("📊 SESSION SUMMARY")
print("="*60)

# Check dataset
if os.path.exists('data/images/train'):
    num_categories = len(os.listdir('data/images/train'))
    print(f"✅ Dataset: MiniPlaces ({num_categories} categories)")
else:
    print("❌ Dataset: Not downloaded")

# Check experiments
exp_folders = sorted(glob.glob('logs/cls_miniplaces_cnn_*'))
if exp_folders:
    print(f"✅ Training experiments: {len(exp_folders)} found")
    latest = exp_folders[-1]
    if os.path.exists(f'{latest}/model_final.pth.tar'):
        print(f"   Latest (complete): {latest}")
    else:
        print(f"   Latest (in progress): {latest}")
else:
    print("❌ Training: No experiments found")

# Check results
if os.path.exists('results/simple_cnn'):
    print(f"✅ Results saved: results/simple_cnn/")
else:
    print("❌ Results: Not yet saved")

print("="*60)